<a href="https://colab.research.google.com/github/YamilaTaschuk/TrabajoPractico/blob/main/YamilaTaschukIntegrador2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Generación de archivos para Frontier Check-in Inteligente
# Autor: Yamila Taschuk
# ============================================================

from google.colab import files

# --- app.py ---
app_code = """import streamlit as st
from PIL import Image, ImageEnhance, ImageFilter, ImageDraw, ImageFont
import numpy as np
from skimage.metrics import structural_similarity as ssim
from transformers import CLIPModel, CLIPImageProcessor, CLIPTokenizer
import torch

st.set_page_config(page_title="Frontier Check-in Inteligente", page_icon="✈️")
st.title("✈️ Frontier Check-in Inteligente")
st.write("Completa tu check-in de Frontier Airlines de manera visual e inteligente.")

# --- Datos del usuario ---
st.subheader("👤 Ingreso de Datos")
nombre = st.text_input("Nombre")
apellido = st.text_input("Apellido")
destino = st.text_input("Destino (Estado de USA)")

estados_usa = ['alabama','alaska','arizona','arkansas','california','colorado','connecticut',
               'delaware','florida','georgia','hawaii','idaho','illinois','indiana','iowa','kansas',
               'kentucky','louisiana','maine','maryland','massachusetts','michigan','minnesota',
               'mississippi','missouri','montana','nebraska','nevada','new hampshire','new jersey',
               'new mexico','new york','north carolina','north dakota','ohio','oklahoma','oregon',
               'pennsylvania','rhode island','south carolina','south dakota','tennessee','texas',
               'utah','vermont','virginia','washington','west virginia','wisconsin','wyoming']

if destino:
    if destino.lower() not in estados_usa:
        st.error("❌ El destino no se encuentra disponible")
    else:
        st.success("✅ Destino válido")

# Cargar modelos CLIP
@st.cache_resource(show_spinner=False)
def load_clip_models():
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    image_processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
    tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    return model, image_processor, tokenizer

clip_model, image_processor, tokenizer = load_clip_models()

def analyze_with_clip(image, labels):
    image_inputs = image_processor(images=image, return_tensors="pt")
    text_inputs = tokenizer(labels, padding=True, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model(pixel_values=image_inputs.pixel_values, input_ids=text_inputs.input_ids)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]
    return sorted(zip(labels, probs), key=lambda x: x[1], reverse=True)

def draw_usa_flag(draw, width, height):
    draw.rectangle([0, 0, width, height], fill="white")
    stripe_height = height // 13
    for i in range(13):
        color = "#BF0A30" if i % 2 == 0 else "white"
        draw.rectangle([0, i*stripe_height, width, (i+1)*stripe_height], fill=color)
    blue_height = stripe_height * 7
    blue_width = int(width * 0.4)
    draw.rectangle([0, 0, blue_width, blue_height], fill="#002868")
    star_radius = 6
    offset_x, offset_y = 15, 15
    for row in range(5):
        for col in range(6):
            x = offset_x + col * 2 * star_radius + (row % 2) * star_radius
            y = offset_y + row * 2 * star_radius
            draw.ellipse([x-star_radius, y-star_radius, x+star_radius, y+star_radius], fill="white")

def generate_ticket_image(nombre, apellido, destino):
    width, height = 512, 256
    ticket = Image.new("RGB", (width, height))
    draw = ImageDraw.Draw(ticket)
    draw_usa_flag(draw, width, height)
    try:
        font = ImageFont.truetype("arial.ttf", 24)
    except:
        font = ImageFont.load_default()

    def draw_text_with_shadow(draw, pos, text, font):
        x, y = pos
        shadow_color = "white"
        text_color = "black"
        offsets = [(1,1), (1,-1), (-1,1), (-1,-1)]
        for ox, oy in offsets:
            draw.text((x+ox, y+oy), text, font=font, fill=shadow_color)
        draw.text((x, y), text, font=font, fill=text_color)

    draw_text_with_shadow(draw, (30, 25), "Frontier Airlines", font)
    draw_text_with_shadow(draw, (30, 70), f"Pasajero: {nombre} {apellido}", font)
    draw_text_with_shadow(draw, (30, 110), f"Destino: {destino.title()}", font)
    draw_text_with_shadow(draw, (30, 160), "Fecha: 20/11/2025", font)
    draw_text_with_shadow(draw, (30, 200), "Código de barra: | | | | | | |", font)
    return ticket

# --- Subida y mejora de documento ---
st.subheader("📄 Subir Documento de Identidad")
uploaded_doc = st.file_uploader("Cargar documento (jpg/png)", type=["jpg","jpeg","png"], key="doc")

if uploaded_doc:
    image = Image.open(uploaded_doc).convert("RGB")
    st.image(image, caption="Imagen original", use_column_width=True)

    # Segmentación simple de bordes
    seg = image.convert("L").filter(ImageFilter.FIND_EDGES)
    st.image(seg, caption="Segmentación de bordes", use_column_width=True)

    # Mejora visual: brillo y contraste
    enhancer_brightness = ImageEnhance.Brightness(image)
    enhanced_brightness = enhancer_brightness.enhance(1.2)
    enhancer_contrast = ImageEnhance.Contrast(enhanced_brightness)
    enhanced_image = enhancer_contrast.enhance(1.5)

    col1, col2 = st.columns(2)
    with col1:
        st.image(image, caption="Antes de mejora")
    with col2:
        st.image(enhanced_image, caption="Después de mejora")

    # Métricas de calidad
    import numpy as np
    orig_arr = np.array(image.convert("L"))
    enh_arr = np.array(enhanced_image.convert("L"))
    psnr_val = 10 * np.log10(255**2 / np.mean((orig_arr - enh_arr) ** 2))
    ssim_val = ssim(orig_arr, enh_arr)
    st.write(f"PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.2f}")

    # Análisis CLIP imagen documento
    labels = ["documento", "foto carnet", "imagen borrosa", "texto ilegible", "paisaje"]
    clip_results_doc = analyze_with_clip(enhanced_image, labels)
    st.subheader("🔍 Análisis CLIP del documento")
    for label, prob in clip_results_doc:
        st.write(f"{label}: {prob:.2%}")

    # Crear y mostrar ticket digital
    if nombre and apellido and destino:
        ticket_img = generate_ticket_image(nombre, apellido, destino)
        st.subheader("🎫 Ticket digital generado")
        st.image(ticket_img, caption="Ticket con bandera USA y datos", width=512)

        # Análisis CLIP ticket generado
        ticket_labels = ["ticket de vuelo", "boarding pass", "documento", "foto carnet", "texto ilegible", "paisaje"]
        clip_results_ticket = analyze_with_clip(ticket_img, ticket_labels)
        st.subheader("🔎 Análisis visual del ticket con CLIP")
        for label, prob in clip_results_ticket:
            st.write(f"{label}: {prob:.2%}")

# --- Subida y mejora de ticket ---
st.subheader("🎫 Subir Ticket de Vuelo")
st.info("Descarga la imagen de su ticket en la app de Frontier Airlines")
uploaded_ticket = st.file_uploader("Cargar ticket (jpg/png)", type=["jpg","jpeg","png"], key="ticket")

if uploaded_ticket:
    ticket = Image.open(uploaded_ticket).convert("RGB")
    st.image(ticket, caption="Ticket original", use_column_width=True)

    enhanced_ticket = ticket.filter(ImageFilter.SHARPEN)
    enhanced_ticket = ImageEnhance.Contrast(enhanced_ticket).enhance(1.5)
    enhanced_ticket = ImageEnhance.Brightness(enhanced_ticket).enhance(1.2)

    col1, col2 = st.columns(2)
    with col1:
        st.image(ticket, caption="Antes del procesamiento")
    with col2:
        st.image(enhanced_ticket, caption="Después del procesamiento")

    # Segmentación simple
    seg_ticket = enhanced_ticket.convert("L").filter(ImageFilter.FIND_EDGES)
    st.image(seg_ticket, caption="Segmentación de bordes", use_column_width=True)

    orig_arr = np.array(ticket.convert("L"))
    enh_arr = np.array(enhanced_ticket.convert("L"))
    psnr_val = 10 * np.log10(255**2 / np.mean((orig_arr - enh_arr) ** 2))
    ssim_val = ssim(orig_arr, enh_arr)
    st.write(f"PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.2f}")

    # Análisis CLIP imagen ticket
    ticket_labels = ["ticket de vuelo", "boarding pass", "documento", "foto carnet", "texto ilegible", "paisaje"]
    clip_results_ticket_uploaded = analyze_with_clip(enhanced_ticket, ticket_labels)
    st.subheader("🔍 Análisis CLIP del ticket subido")
    for label, prob in clip_results_ticket_uploaded:
        st.write(f"{label}: {prob:.2%}")

if uploaded_doc and uploaded_ticket:
    st.success("✅ Su check-in está listo, ¡buen viaje!")
"""

# --- requirements.txt ---
reqs = """streamlit
torch
diffusers>=0.15.1
transformers>=4.32.0
huggingface_hub>=0.27.0,<1.0
Pillow
regex
numpy
scikit-image
"""

# --- README.md ---
readme = """---
title: Frontier Check-in Inteligente
emoji: ✈️
colorFrom: blue
colorTo: green
sdk: streamlit
sdk_version: "1.26.1"
app_file: app.py
pinned: false
---

# ✈️ Frontier Check-in Inteligente

## 🎯 Objetivo
Aplicación web que ayuda a los usuarios a completar su check-in de Frontier Airlines de manera visual e inteligente.
Mejora la calidad de las imágenes de documento y ticket usando técnicas básicas de mejora visual con Pillow, realiza segmentación simple, y analiza imágenes semánticamente con modelo CLIP.

## 👤 User Persona
**Nombre:** Yamila Taschuk
**Edad:** 35 años
**Necesidad:** Validar que su documento de identidad y ticket estén legibles antes del check-in.

## 🏗️ Arquitectura
Usuario → Upload de documento → Mejora visual y segmentación simple → Upload de ticket → Mejora y segmentación → Análisis semántico con CLIP → Resultado visual y métricas

## 🛠️ Stack
- Streamlit
- Pillow
- Torch
- Transformers (CLIP)
- scikit-image
- numpy

## ⚡ Funcionalidades
- Ingreso y validación de datos personales
- Mejora visual básica (brillo y contraste)
- Segmentación simple de bordes
- Análisis semántico con CLIP de documentos y tickets
- Visualización lado a lado y métricas PSNR/SSIM
- Generación visual sencilla de ticket digital con bandera USA y datos usuario

## 📚 Conceptos PDI aplicados
- Restauración básica de imagen (filtros Pillow)
- Segmentación (detección de bordes)
- Clasificación visual semántica con CLIP
- Comparación visual y métricas de calidad

## ⚠️ Limitaciones
- Sin modelos de difusión ni restauración avanzada
- No OCR ni validación legal
- Segmentación y mejora simples, optimizadas para CPU
- Validación solo para destinos dentro de USA

## 👨‍💻 Autor
Yamila Taschuk
IFT S24 – Tecnicatura en Ciencias de Datos e IA (2025)

## 📸 Ejemplo de Uso
- Ingreso de datos personales y validación:
![Ingreso Nombre, apellido y Destino](validacion.png)
- Upload, mejora y segmentación de documento: ![Ticket Pasaporte](documentacion.png)
- Upload, mejora y segmentación de ticket: ![Ticket Ejemplo](ticket.png)
- Análisis CLIP en ambas imágenes : ![Clip Documentacion](clipdocu.png)
![Clip ticket](clipticket.png)
- Visualización de ticket digital con bandera USA y datos: ![Ticket digital](usa.png)

🎥 **Video demostrativo del proyecto**
[Haz clic aquí para ver el video](https://drive.google.com/file/d/1FBaG7DZGsGRYEXzHLZzB6X5AAfIFGay-/view?usp=sharing)























"""

# Guardar archivos
with open("app.py","w") as f:
    f.write(app_code)
with open("requirements.txt","w") as f:
    f.write(reqs)
with open("README.md","w") as f:
    f.write(readme)

# Descargar archivos
files.download("app.py")
files.download("requirements.txt")
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>